# Lab 2: Create a Simple AI Agent

In this lab, we'll introduce you to AI agents by creating a simple agent using Microsoft Foundry Agent SDK, that will create a bar graph based on data that we give to it.

#### Step 1: Load packages

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential
from azure.ai.agents.models import CodeInterpreterTool

load_dotenv()  # Load environment variables from .env file

#### Step 2: Connect to your Microsoft Foundry Project
Use a token credential for the Agents SDK. In this cell, the code tries `AzureCliCredential` first (after `az login`) and falls back to `DefaultAzureCredential`.

In [ ]:
# Connect to Microsoft Foundry project using DefaultAzureCredential, a type of token-based authentication.
project = AIProjectClient(
    endpoint=os.getenv("AIPROJECT_ENDPOINT"),
    credential=DefaultAzureCredential()
)

#### Step 3: Create the simple AI Agent and Chat with the Agent

This cell demonstrates how to use the Microsoft Foundry SDK to create and interact with an AI agent enhanced with the Code Interpreter tool. The workflow includes:

- Initializing a Code Interpreter tool and creating an agent with it.
- Creating a conversation to communicate with the agent.
- Sending health plan data to the agent and receiving a recommendation, followed by a summary in a subsequent interaction.
- Generating a health plan comparison bar chart locally using Python and Matplotlib, and saving it as a PNG file.
- Cleaning up by deleting the agent after the workflow completes.

This exercise demonstrates how AI agents can assist with analysing structured data through conversational interactions while Python is used to create and save visualisations for reporting.

In [ ]:
import matplotlib.pyplot as plt

code_interpreter = CodeInterpreterTool()

# Save single output in the opened workspace folder
output_dir = Path.cwd()
chart_path = output_dir / "health-plan-report.png"

# Source data used for chart
providers = ["Northwind", "Aetna", "United Health", "Premera"]
monthly_premiums = [300, 350, 250, 200]
deductibles = [1500, 1000, 2000, 2200]
oop_limits = [6000, 5500, 7000, 6500]

with project:
    # New Foundry SDK: create agent version instead of create_agent
    agent_name = os.getenv("AGENT_NAME", "my-agent")
    agent_definition = PromptAgentDefinition(
        model=os.environ["CHAT_MODEL"],
        instructions="You are a helpful health plan advisor.",
    )

    # Attach Code Interpreter when supported by this SDK build
    if hasattr(agent_definition, "tools"):
        agent_definition.tools = code_interpreter.definitions
    else:
        print("Warning: This SDK build does not expose agent_definition.tools.")

    agent = project.agents.create_version(
        agent_name=agent_name,
        definition=agent_definition,
    )
    print(f"Created agent, ID: {agent.id}")

    # New Foundry SDK: conversations + responses (thread/message/run equivalent)
    openai = project.get_openai_client(agent_name=agent_name)
    conversation = openai.conversations.create()
    print(f"Created thread, ID: {conversation.id}")

    # First turn
    response = openai.responses.create(
        conversation=conversation.id,
        input=(
            "Given this health-plan data, provide a short recommendation.\n\n"
            "Provider        Monthly Premium    Deductible    Out-of-Pocket Limit\n"
            "Northwind       $300               $1,500        $6,000\n"
            "Aetna           $350               $1,000        $5,500\n"
            "United Health   $250               $2,000        $7,000\n"
            "Premera         $200               $2,200        $6,500\n"
        ),
    )
    print("Created message, ID: response-generated")
    print("Run finished with status: completed")
    print("Conversation:")
    print(f"assistant: {response.output_text}")

    # Follow-up turn (printed only, no report file)
    follow_up = openai.responses.create(
        conversation=conversation.id,
        input="Summarize your recommendation in 3 bullet points.",
    )
    print(f"assistant: {follow_up.output_text}")

    # Generate chart like image 2 and save as a single PNG output file
    x = range(len(providers))
    width = 0.25

    plt.figure(figsize=(12, 7))
    plt.bar(
        [i - width for i in x],
        monthly_premiums,
        width=width,
        label="Monthly Premium",
        color="#1f77b4",
    )
    plt.bar(
        x,
        deductibles,
        width=width,
        label="Deductible",
        color="#ff7f0e",
    )
    plt.bar(
        [i + width for i in x],
        oop_limits,
        width=width,
        label="Out-of-Pocket Limit",
        color="#2ca02c",
    )

    plt.title("Health Plan Comparison")
    plt.xlabel("Provider")
    plt.ylabel("Amount ($)")
    plt.xticks(list(x), providers, rotation=15)
    plt.legend()
    plt.tight_layout()
    plt.savefig(chart_path, dpi=150)
    plt.close()
    print(f"Saved chart to: {chart_path}")
    print("Only one output file is created: health-plan-report.png")

    # Delete agent resource (compatible with SDK minor differences)
    deleted = False
    delete_agent = getattr(project.agents, "delete_agent", None)
    if callable(delete_agent):
        try:
            delete_agent(agent.id)
            deleted = True
        except TypeError:
            try:
                delete_agent(agent_id=agent.id)
                deleted = True
            except Exception:
                pass

    if not deleted:
        delete_version = getattr(project.agents, "delete_version", None)
        if callable(delete_version):
            try:
                delete_version(agent_name=agent_name, agent_version=agent.version)
                deleted = True
            except TypeError:
                try:
                    delete_version(agent_name, agent.version)
                    deleted = True
                except Exception:
                    pass

    if deleted:
        print("Deleted agent")
    else:
        print("Cleanup skipped: delete API not available in this SDK version.")